# Baseline Model: IoT-23 Binary Classification

In [1]:
import numpy as np
import pandas as pd
import random
from pathlib import Path

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_PATH = Path("../data/iot23_processed/iot23_binary_flows.csv")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok = True)

### 1) Dataset Loading and Cleaning
##### Loads the IoT-23 processed flow dataset. Missing values represented as '-' are converted to NaN and removed to ensure numerical compatibility with machine learning models


In [2]:
# Load and clean the data
df = pd.read_csv(DATA_PATH)

# Replace Zeek placeholder "-" with NA
df.replace("-", pd.NA, inplace = True)

# Convert numeric columns
numeric_cols = [
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts",
    "orig_ip_bytes",
    "resp_ip_bytes"
]

# Convert columns to numeric, coercing errors to NaN
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors = "coerce")

# Drop rows with missing numeric values or missing label
df.dropna(subset = numeric_cols + ["binary_label"], inplace = True)

# Reset index after dropping rows
df.reset_index(drop = True, inplace = True)

print("After cleaning:", df.shape)
print(df["binary_label"].value_counts())

After cleaning: (2291, 13)
binary_label
BENIGN       1527
MALICIOUS     764
Name: count, dtype: int64


### 2) Feature Selection 
##### Flow-level numerical features selected and encode the binary label as 0 for benign and 1 for malicious

In [3]:
# Prepare features and labels
feature_cols = [
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts",
]

# Drop rows with missing values in features or label
df = df.dropna(subset = feature_cols + ["binary_label"]).copy()
X = df[feature_cols]
# Convert binary label to 0 for benign and 1 for malicious
y = (df["binary_label"] == "MALICIOUS").astype(int)

print("X:", X.shape, "y:", y.shape)
print(y.value_counts())

X: (2291, 5) y: (2291,)
binary_label
0    1527
1     764
Name: count, dtype: int64


### 3) Encode Categorical Features
##### Network protocol, service type and connection state are categorical features so we convert them into numeric format

In [4]:
# Encode categorical variables using one-hot encoding
categorical_cols = ["proto", "service", "conn_state"]
df_encoded = pd.get_dummies(df, columns = categorical_cols, drop_first = True)
print("After encoding:", df_encoded.shape)

After encoding: (2291, 24)


### 4) Define Feature and Target Variables
##### Separate input features (X) from the binary target label (y)

In [5]:
# Prepare features and target variable
feature_cols = [
    col for col in df_encoded.columns
    if col not in ["binary_label", "ts", "capture_name"]
]

# Create feature matrix X and target vector y
X = df_encoded[feature_cols]
y = (df_encoded["binary_label"] == "MALICIOUS").astype(int)

print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Feature matrix shape: (2291, 21)
Target distribution:
 binary_label
0    1527
1     764
Name: count, dtype: int64


### 5) Train-test Split
##### Dataset is split into training and testing subsets using a 70/30 split to preserve class distribution

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (1603, 21)
Test size: (688, 21)


### 6) Random Forest Model Training
##### A Random Forest classifier is trained as the baseline model due to its robustness and ability to handle non-linear decision boundaries

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Random Forest
rf = RandomForestClassifier(n_estimators = 200, random_state = 42, n_jobs = -1)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

Random Forest Results:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       459
           1       1.00      0.98      0.99       229

    accuracy                           0.99       688
   macro avg       1.00      0.99      0.99       688
weighted avg       0.99      0.99      0.99       688

ROC-AUC: 0.9999143762308417


### 7) Logistic Regression Baseline
##### Logistic Regression classifier is trained and evaluated under the same conditions as the Random Forest model

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Logistic Regression with scaling (for XGBoost)
log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter = 5000, random_state = 42))
])

log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)[:, 1]

print("\nLogistic Regression Results:")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))


Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       459
           1       1.00      0.98      0.99       229

    accuracy                           0.99       688
   macro avg       0.99      0.99      0.99       688
weighted avg       0.99      0.99      0.99       688

ROC-AUC: 0.9895729276669426


### 8) Decision Tree Classifier
##### A single Decision Tree model is trained to compare against ensemble methods

In [9]:
from sklearn.tree import DecisionTreeClassifier
import time

# Measure training time
start = time.time()
dt = DecisionTreeClassifier(random_state = 42)
dt.fit(X_train, y_train)
dt_train_time = time.time() - start

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]

print("Decision Tree Results:")
print(classification_report(y_test, y_pred_dt))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_dt))
print("Training time: {:.4f} seconds".format(dt_train_time))

Decision Tree Results:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       459
           1       1.00      0.98      0.99       229

    accuracy                           0.99       688
   macro avg       0.99      0.99      0.99       688
weighted avg       0.99      0.99      0.99       688

ROC-AUC: 0.9879936448135781
Training time: 0.0050 seconds


### 9) XGBoost Classifier
##### XGBoost is trained under identical conditions to compare boosting performance against Random Forest and single-tree models

In [10]:
from xgboost import XGBClassifier
import time

# Start timing
start = time.time()

xgb = XGBClassifier(
    n_estimators = 200,
    max_depth = 6,
    learning_rate = 0.1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    random_state = 42,
    eval_metric = "logloss",
    n_jobs = -1
)

xgb.fit(X_train, y_train)

xgb_train_time = time.time() - start

y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))
print("Training time: {:.4f} seconds".format(xgb_train_time))

XGBoost Results:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       459
           1       1.00      0.99      0.99       229

    accuracy                           1.00       688
   macro avg       1.00      0.99      1.00       688
weighted avg       1.00      1.00      1.00       688

ROC-AUC: 0.9984492584030216
Training time: 0.0706 seconds


### LightGBM Model
##### LightGBM is an efficient gradient boosting framework designed for high performance and low memory usage and is particularly suited for large-scale and resource-constrained environments

In [11]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Initialise LightGBM classifier
lgbm = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)

# Train model
lgbm.fit(X_train, y_train)

# Predictions
y_pred_lgbm = lgbm.predict(X_test)
y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]

# Evaluation
print("LightGBM Results:")
print(classification_report(y_test, y_pred_lgbm))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lgbm))


[LightGBM] [Info] Number of positive: 535, number of negative: 1068
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000281 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 583
[LightGBM] [Info] Number of data points in the train set: 1603, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333749 -> initscore=-0.691276
[LightGBM] [Info] Start training from score -0.691276
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

### Comparison Table

In [12]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

results = {
    "Model": [
        "Decision Tree",
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_lgbm)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_lgbm)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_dt),
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_xgb),
        roc_auc_score(y_test, y_proba_lgbm)
    ]
}

comparison_df = pd.DataFrame(results)
comparison_df

,Model,Accuracy,F1 Score,ROC-AUC
0,Decision Tree,0.991279,0.986784,0.987994
1,Logistic Regression,0.992733,0.988962,0.989573
2,Random Forest,0.994186,0.991189,0.999914
3,XGBoost,0.995640,0.993407,0.998449
4,LightGBM,0.998547,0.997812,0.999429


### Model sizes

In [13]:
# Will later be used to save the models and report their sizes
import joblib
import os
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

rf_path = models_dir / "rf_model.joblib"
lr_path = models_dir / "log_reg_model.joblib"
dt_path = models_dir / "decision_tree_model.joblib"
xgb_path = models_dir / "xgb_model.joblib"
lgbm_path = models_dir / "lgbm_model.joblib"

joblib.dump(lgbm, lgbm_path)
joblib.dump(dt, dt_path)
joblib.dump(rf, rf_path)
joblib.dump(log_reg, lr_path)
joblib.dump(xgb, xgb_path)

rf_size = os.path.getsize(rf_path) / 1024
lr_size = os.path.getsize(lr_path) / 1024
dt_size = os.path.getsize(dt_path) / 1024
xgb_size = os.path.getsize(xgb_path) / 1024
lgbm_size = os.path.getsize(lgbm_path) / 1024

print(f"Random Forest size: {rf_size:.2f} KB")
print(f"Logistic Regression size: {lr_size:.2f} KB")
print(f"Decision Tree size: {dt_size:.2f} KB")
print(f"XGBoost size: {xgb_size:.2f} KB")
print(f"LightGBM size: {lgbm_size:.2f} KB")

Random Forest size: 598.04 KB
Logistic Regression size: 2.49 KB
Decision Tree size: 4.06 KB
XGBoost size: 181.76 KB
LightGBM size: 392.10 KB
